In [16]:
from pymongo import MongoClient

# Membuat koneksi ke MongoDB yang berjalan di localhost, port default 27017
client = MongoClient("mongodb://localhost:27017/")

# Memilih (atau otomatis membuat) database
db = client["kopi_nusantara"]

# Memilih (atau otomatis membuat) collection
koleksi_menu = db["menu"]

print("Koneksi ke MongoDB berhasil!")
print("Daftar database yang ada:", client.list_database_names())

Koneksi ke MongoDB berhasil!
Daftar database yang ada: ['admin', 'config', 'kopi_nusantara', 'local', 'praktikum_bigdata']


In [4]:
koleksi_menu.insert_many([
    {"nama_menu": "Americano", "harga": 17000, "kategori": "Kopi", "tingkat_kepahitan": 5},
    {"nama_menu": "Cappuccino", "harga": 21000, "kategori": "Kopi", "tingkat_kepahitan": 2},
    {"nama_menu": "Latte", "harga": 21000, "kategori": "Kopi", "tingkat_kepahitan": 2},
    {"nama_menu": "Matcha Espresso", "harga": 25000, "kategori": "Kopi", "tingkat_kepahitan": 5},
    {"nama_menu": "Avocado Coffee", "harga": 26000, "kategori": "Kopi", "tingkat_kepahitan": 3},
    {"nama_menu": "Creamy Aren Latte", "harga": 21000, "kategori": "Kopi", "tingkat_kepahitan": 2},
    {"nama_menu": "Nasi Goreng", "harga": 17000, "kategori": "Makanan", "pedas_level": 2},
    {"nama_menu": "Mie Goreng", "harga": 16000, "kategori": "Makanan", "pedas_level": 1},
    {"nama_menu": "Mie Kwetiaw", "harga": 18000, "kategori": "Makanan", "pedas_level": 1},
    {"nama_menu": "Nasi Briyani", "harga": 25000, "kategori": "Makanan", "pedas_level": 2},
    {"nama_menu": "Nasi Padang", "harga": 17000, "kategori": "Makanan", "pedas_level": 2},
    {"nama_menu": "Nasi Rawon", "harga": 15000, "kategori": "Makanan", "pedas_level": 2},
])

print("Jumlah dokumen di collection 'menu':", koleksi_menu.count_documents({}))    

Jumlah dokumen di collection 'menu': 12


In [39]:
# Seluruh menu kategori "Kopi" dengan tingkat_kepahitan di atas 3
print("Kopi dengan tingkat kepahitan di atas 3:")
for m in koleksi_menu.find({"tingkat_kepahitan": {"$gt": 3}}):
    print(f"  - {m['nama_menu']} ({m['tingkat_kepahitan']:,})")

# Seluruh menu dengan harga di antara 15000 dan 30000.
print("\nProduk dengan harga di antara 15000 dan 30000:")
for m in koleksi_menu.find({"harga": {"$gt": 15000, "$lt": 30000}}):
    print(f"  - {m['nama_menu']} (Rp{m['harga']:,})")

# Menu yang tidak memiliki field pedas_level sama sekali.
print("\nMenu yang tidak memiliki pedas_level:")
for m in koleksi_menu.find({"pedas_level": {"$exists": False}}):
    print(f"  - {m['nama_menu']}")

Kopi dengan tingkat kepahitan di atas 3:
  - Americano (5)
  - Matcha Espresso (5)

Produk dengan harga di antara 15000 dan 30000:
  - Americano (Rp17,000)
  - Cappuccino (Rp21,000)
  - Latte (Rp21,000)
  - Matcha Espresso (Rp25,000)
  - Avocado Coffee (Rp26,000)
  - Creamy Aren Latte (Rp21,000)
  - Nasi Goreng (Rp17,000)
  - Mie Goreng (Rp16,000)
  - Mie Kwetiaw (Rp18,000)
  - Nasi Briyani (Rp25,000)
  - Nasi Padang (Rp17,000)

Menu yang tidak memiliki pedas_level:
  - Americano
  - Cappuccino
  - Latte
  - Matcha Espresso
  - Avocado Coffee
  - Creamy Aren Latte


In [53]:
# Tambahkan field status bernilai "Tersedia" ke seluruh dokumen menggunakan
# update_many(). Kemudian, ubah status menjadi "Habis" khusus untuk 2 menu 
# pilihan menggunakan update_one()

koleksi_menu.update_many(
    {},
    {"$set": {"status": "Tersedia"}}
)
print("Setelah update:")
for m in koleksi_menu.find({}, {"_id":0, "nama_menu": 1, "harga": 1, "status": 1}):
    print(f"  - {m['nama_menu']}: harga=Rp{m['harga']:,}, status={m['status']}")


koleksi_menu.update_one(
    {"nama_menu": "Latte"},
    {"$set": {"status": "Habis"}}
)

koleksi_menu.update_one(
    {"nama_menu": "Matcha Espresso"},
    {"$set": {"status": "Habis"}}
)

koleksi_menu.update_one(
    {"nama_menu": "Nasi Briyani"},
    {"$set": {"status": "Habis"}}
)

print("\nSetelah update patch:")
for m in koleksi_menu.find({}, {"_id":0, "nama_menu": 1, "harga": 1, "status": 1}):
    print(f"  - {m['nama_menu']}: harga=Rp{m['harga']:,}, status={m['status']}")

Setelah update:
  - Americano: harga=Rp17,000, status=Tersedia
  - Cappuccino: harga=Rp21,000, status=Tersedia
  - Latte: harga=Rp21,000, status=Tersedia
  - Matcha Espresso: harga=Rp25,000, status=Tersedia
  - Avocado Coffee: harga=Rp26,000, status=Tersedia
  - Creamy Aren Latte: harga=Rp21,000, status=Tersedia
  - Nasi Goreng: harga=Rp17,000, status=Tersedia
  - Mie Goreng: harga=Rp16,000, status=Tersedia
  - Mie Kwetiaw: harga=Rp18,000, status=Tersedia
  - Nasi Briyani: harga=Rp25,000, status=Tersedia
  - Nasi Padang: harga=Rp17,000, status=Tersedia
  - Nasi Rawon: harga=Rp15,000, status=Tersedia

Setelah update patch:
  - Americano: harga=Rp17,000, status=Tersedia
  - Cappuccino: harga=Rp21,000, status=Tersedia
  - Latte: harga=Rp21,000, status=Habis
  - Matcha Espresso: harga=Rp25,000, status=Habis
  - Avocado Coffee: harga=Rp26,000, status=Tersedia
  - Creamy Aren Latte: harga=Rp21,000, status=Tersedia
  - Nasi Goreng: harga=Rp17,000, status=Tersedia
  - Mie Goreng: harga=Rp16,00

In [57]:
# Integrasi ke Spark
from pyspark.sql import SparkSession
import pandas as pd

spark = SparkSession.builder \
    .appName("Pertemuan9-MongoDBSpark") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

# Langkah 1: Ambil seluruh dokumen dari MongoDB sebagai list of dict
file_mongo = list(koleksi_menu.find({}))

# Langkah 2: Ubah menjadi pandas DataFrame
df_pandas = pd.DataFrame(file_mongo)

# Langkah 3: Tipe data khusus MongoDB (_id, dan kolom berisi list seperti "tags")
# perlu disesuaikan dulu agar Spark dapat membacanya dengan benar
df_pandas["_id"] = df_pandas["_id"].astype(str)
# df_pandas["tingkat_kepahitan"] = df_pandas["tingkat_kepahitan"].apply(lambda x: ", ".join(x) if isinstance(x, list) else "")

# Langkah 4: Konversi ke Spark DataFrame
df_spark_menu = spark.createDataFrame(df_pandas)

print("1. Print Schema")
df_spark_menu.printSchema()
df_spark_menu.show()




1. Print Schema
root
 |-- _id: string (nullable = true)
 |-- nama_menu: string (nullable = true)
 |-- harga: long (nullable = true)
 |-- kategori: string (nullable = true)
 |-- tingkat_kepahitan: double (nullable = true)
 |-- status: string (nullable = true)
 |-- pedas_level: double (nullable = true)



+--------------------+-----------------+-----+--------+-----------------+--------+-----------+
|                 _id|        nama_menu|harga|kategori|tingkat_kepahitan|  status|pedas_level|
+--------------------+-----------------+-----+--------+-----------------+--------+-----------+
|6aeaa9eaf09d104a9...|        Americano|17000|    Kopi|              5.0|Tersedia|        NaN|
|6aeaa9eaf09d104a9...|       Cappuccino|21000|    Kopi|              2.0|Tersedia|        NaN|
|6aeaa9eaf09d104a9...|            Latte|21000|    Kopi|              2.0|   Habis|        NaN|
|6aeaa9eaf09d104a9...|  Matcha Espresso|25000|    Kopi|              5.0|   Habis|        NaN|
|6aeaa9eaf09d104a9...|   Avocado Coffee|26000|    Kopi|              3.0|Tersedia|        NaN|
|6aeaa9eaf09d104a9...|Creamy Aren Latte|21000|    Kopi|              2.0|Tersedia|        NaN|
|6aeaa9eaf09d104a9...|      Nasi Goreng|17000| Makanan|              NaN|Tersedia|        2.0|
|6aeaa9eaf09d104a9...|       Mie Goreng|16000| Mak

In [56]:
from pyspark.sql.functions import avg, count

print("2. Rata-rata harga per kategori")
df_spark_menu.groupBy("kategori").agg(
    avg("harga").alias("rata_rata_harga"),
    count("nama_menu").alias("jumlah_menu")
).show()

spark.stop()
print("SparkSession ditutup.")

2. Rata-rata harga per kategori


+--------+------------------+-----------+
|kategori|   rata_rata_harga|jumlah_menu|
+--------+------------------+-----------+
|    Kopi|21833.333333333332|          6|
| Makanan|           18000.0|          6|
+--------+------------------+-----------+

SparkSession ditutup.


In [ ]:
Tulis pada markdown cell (minimal 100 kata): apa tantangan yang anda temui saat mengonversi data MongoDB (dengan field yang tidak seragam 
antar dokumen) menjadi Spark DataFrame (yang mengharuskan skema tabel seragam)? Bagaimana anda mengatasinya?

Tantangan terbesar yang saya hadapi terjadi pada tahap konversi data dari MongoDB ke dalam Apache Spark DataFrame, di mana kolom 
tingkat_kepahitan sempat tidak muncul dengan benar. Setelah dilakukan penelusuran, ternyata kendala tersebut bersumber dari kesalahan 
kecil pada skrip pembersihan data (data cleaning) saat menangani tipe data berbentuk list dan struktur dokumen yang dinamis. Pengalaman 
ini memberikan saya pemahaman mendalam mengenai krusialnya tahap preprocessing sebelum data dari basis data NoSQL diintegrasikan ke 
ekosistem Big Data seperti PySpark. Karena MongoDB bersifat fleksibel di mana dokumen kategori Kopi dan Makanan memiliki atribut unik yang 
berbeda, seperti tingkat_kepahitan dan pedas_level, penanganan nilai kosong (null) serta konversi tipe data yang tepat menjadi penentu 
keberhasilan analisis. Setelah logika pembersihan tersebut saya perbaiki, proses konversi berjalan lancar dan seluruh data berhasil 
ditampilkan dengan aman serta akurat.